# Transformer_Toy — Shakedown Comparison Script

This notebook exercises the complete Python API over the native C++/CUDA implementation:

- BPE tokenizer persistence
- full-corpus-equivalent training batches
- configurable Transformer construction
- Adam optimization
- CPU/CUDA selection
- validation
- cosine learning-rate decay
- early stopping
- periodic and best checkpoints
- generation snapshots every five epochs
- final best-checkpoint generation

The default run is identical the native shakedown: **25 epochs**, logging every **100 steps**, generation every **5 epochs**, and approximately one corpus worth of training tokens per epoch. The settings are completely identical to the native shakedown, providing a good comparison for performance.

## 1. Environment

Run from the repository root with `.venv` active. Build the extension first:

```powershell
cmake --build build --config Release --target _transformer_toy
$env:PYTHONPATH = "$PWD\python"
```

If necessary, register ipkernel interface:

```powershell
python -m ipykernel install --user --name transformer-toy --display-name "Python (Transformer_Toy)"
```

In [2]:
from __future__ import annotations

from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import transformer_toy as tt

print("CUDA available:", tt.cuda_available())

CUDA available: True


## 2. Configuration

In [ ]:
PROJECT_ROOT = Path.cwd()
CORPUS_PATH = PROJECT_ROOT / "data" / "shakespeare_complete_cleaned.txt"

ARTIFACT_ROOT = PROJECT_ROOT / "dropout_shakedown" / "drop_bpe_512"
TOKENIZER_PATH = (
    PROJECT_ROOT
    / "tokenizers"
    / "complete_shakespeare_bpe_512.tok"
)
CHECKPOINT_DIR = ARTIFACT_ROOT / "checkpoints"

TOKENIZER_PATH.parent.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

DATA_SEED = 2026
MODEL_SEED = 42
GENERATION_SEED = 2026

CONTEXT_LENGTH = 64
BATCH_SIZE = 16
EPOCHS = 25

D_MODEL = 64
D_FF = 128
NUM_LAYERS = 2
NUM_HEADS = 4

INITIAL_LEARNING_RATE = 1.0e-3
MINIMUM_LEARNING_RATE = 1.0e-5
WEIGHT_DECAY = 0.0
DROPOUT = 0.1

VALIDATION_FRACTION = 0.10
LOG_EVERY_STEPS = 100
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 1.0e-4
CHECKPOINT_EVERY_EPOCHS = 1
GENERATION_EVERY_EPOCHS = 5

GENERATION_PROMPT = "To be"
GENERATION_MAX_NEW_TOKENS = 80
GENERATION_TEMPERATURE = 0.9
GENERATION_TOP_K = 10

if not tt.cuda_available():
    raise RuntimeError(
        "CUDA is required for the C++/Python comparison."
    )

DEVICE = tt.Device.CUDA

print("Device:", DEVICE)
print("Corpus:", CORPUS_PATH)
print("Artifacts:", ARTIFACT_ROOT)

## 3. Build shared BPE tokenizer and datasets

In [ ]:
tokenizer_config = tt.TokenizerConfig()
tokenizer_config.type = tt.TokenizerType.BPE
tokenizer_config.vocab_size = 512
tokenizer_config.min_frequency = 2
tokenizer_config.model_path = str(TOKENIZER_PATH)
tokenizer_config.train_if_missing = True
tokenizer_config.preserve_whitespace = True
tokenizer_config.preserve_punctuation = True

## 4. Build deterministic sequential batches

In [ ]:
from dataclasses import dataclass

@dataclass
class SequentialBatchBuildResult:
    batches: list[tt.TrainingBatch]
    sequence_count: int
    input_token_count: int
    transition_count: int
    tail_sequence_length: int


def build_sequential_batches(
    token_ids: list[int],
    range_begin: int,
    range_end: int,
    context_length: int,
    batch_size: int,
) -> SequentialBatchBuildResult:
    if context_length <= 0:
        raise ValueError(
            "context_length must be greater than zero."
        )

    if batch_size <= 0:
        raise ValueError(
            "batch_size must be greater than zero."
        )

    if range_begin < 0:
        raise ValueError(
            "range_begin cannot be negative."
        )

    if range_begin >= range_end:
        raise ValueError(
            "Sequential batch range must be non-empty."
        )

    if range_end > len(token_ids):
        raise IndexError(
            "Sequential batch range exceeds token count."
        )

    range_token_count = range_end - range_begin

    if range_token_count < 2:
        raise ValueError(
            "Sequential batch range must contain at least two tokens."
        )

    usable_transitions = range_token_count - 1
    full_sequence_count = (
        usable_transitions // context_length
    )
    tail_length = (
        usable_transitions % context_length
    )

    # Convert the relevant token range once.
    range_tokens = np.asarray(
        token_ids[range_begin:range_end],
        dtype=np.float32,
    )

    full_transition_count = (
        full_sequence_count * context_length
    )

    batches: list[tt.TrainingBatch] = []

    # ----------------------------------------------------------
    # Build all full-length sequences with vectorized reshaping.
    # ----------------------------------------------------------
    if full_sequence_count > 0:
        all_inputs = range_tokens[
            :full_transition_count
        ].reshape(
            full_sequence_count,
            context_length,
        )

        all_targets = range_tokens[
            1:full_transition_count + 1
        ].reshape(
            full_sequence_count,
            context_length,
        )

        # This loop runs once per batch, not once per sequence.
        for sequence_start in range(
            0,
            full_sequence_count,
            batch_size,
        ):
            sequence_end = min(
                sequence_start + batch_size,
                full_sequence_count,
            )

            # Make contiguous copies before handing data to C++.
            batch_inputs = np.ascontiguousarray(
                all_inputs[
                    sequence_start:sequence_end
                ]
            )

            batch_targets = np.ascontiguousarray(
                all_targets[
                    sequence_start:sequence_end
                ]
            )

            batches.append(
                tt.TrainingBatch(
                    inputs=tt.Tensor(batch_inputs),
                    targets=tt.Tensor(batch_targets),
                )
            )

    # ----------------------------------------------------------
    # Preserve the shortened final sequence exactly as C++ does.
    # ----------------------------------------------------------
    if tail_length > 0:
        tail_start = full_transition_count

        tail_inputs = np.ascontiguousarray(
            range_tokens[
                tail_start:
                tail_start + tail_length
            ].reshape(1, tail_length)
        )

        tail_targets = np.ascontiguousarray(
            range_tokens[
                tail_start + 1:
                tail_start + tail_length + 1
            ].reshape(1, tail_length)
        )

        batches.append(
            tt.TrainingBatch(
                inputs=tt.Tensor(tail_inputs),
                targets=tt.Tensor(tail_targets),
            )
        )

    sequence_count = (
        full_sequence_count
        + (1 if tail_length > 0 else 0)
    )

    transition_count = (
        full_transition_count + tail_length
    )

    if not batches:
        raise RuntimeError(
            "Sequential batch builder produced no batches."
        )

    if transition_count != usable_transitions:
        raise RuntimeError(
            "Sequential batch builder did not cover all "
            "corpus transitions."
        )

    return SequentialBatchBuildResult(
        batches=batches,
        sequence_count=sequence_count,
        input_token_count=transition_count,
        transition_count=transition_count,
        tail_sequence_length=tail_length,
    )


start = time.perf_counter()

dataset = tt.TextDataset(
    file_path=str(CORPUS_PATH),
    context_length=CONTEXT_LENGTH,
    tokenizer_config=tokenizer_config,
)

tokenizer = dataset.tokenizer

all_token_ids = tokenizer.encode(
    dataset.raw_text
)

if len(all_token_ids) < 100:
    raise RuntimeError(
        "Dataset is unexpectedly small for the shakedown."
    )

split_index = int(
    len(all_token_ids) * 0.90
)

split_index = max(
    2,
    min(
        split_index,
        len(all_token_ids) - 2,
    ),
)

training_build = build_sequential_batches(
    token_ids=all_token_ids,
    range_begin=0,
    range_end=split_index,
    context_length=CONTEXT_LENGTH,
    batch_size=BATCH_SIZE,
)

validation_build = build_sequential_batches(
    token_ids=all_token_ids,
    range_begin=split_index,
    range_end=len(all_token_ids),
    context_length=CONTEXT_LENGTH,
    batch_size=BATCH_SIZE,
)

train_batches = training_build.batches
validation_batches = validation_build.batches

elapsed = time.perf_counter() - start

print("[DATASET]")
print(f"  raw characters: {len(dataset.raw_text):,}")
print(f"  encoded tokens: {len(all_token_ids):,}")
print(f"  vocabulary: {dataset.vocab_size:,}")
print(
    "  compression ratio:",
    len(dataset.raw_text) / len(all_token_ids),
)
print(f"  training tokens: {split_index:,}")
print(f"  training batches: {len(train_batches):,}")
print(f"  validation batches: {len(validation_batches):,}")
print(
    f"  training sequences: "
    f"{training_build.sequence_count:,}"
)
print(
    f"  validation sequences: "
    f"{validation_build.sequence_count:,}"
)
print(
    f"  train transitions covered: "
    f"{training_build.transition_count:,}"
)
print(
    f"  validation transitions covered: "
    f"{validation_build.transition_count:,}"
)
print(
    f"  train tail length: "
    f"{training_build.tail_sequence_length}"
)
print(
    f"  validation tail length: "
    f"{validation_build.tail_sequence_length}"
)
print(f"Batch generation:   {elapsed:.2f} seconds")

## 5. Construct the Transformer

In [ ]:
block_config = tt.TransformerBlockConfig(
    d_model=D_MODEL,
    d_ff=D_FF,
    residual_dropout=DROPOUT,
    ffn_dropout=DROPOUT,
    pre_norm=True,
    use_bias=True,
)
block_config.attention_type = tt.AttentionType.MULTI_HEAD
block_config.num_heads = NUM_HEADS
block_config.validate()

model_config = tt.TransformerModelConfig(
    vocab_size=dataset.vocab_size,
    max_seq_len=CONTEXT_LENGTH,
    num_layers=NUM_LAYERS,
    block=block_config,
    learned_positional_embeddings=True,
    embedding_dropout=DROPOUT,
)
model_config.validate()

model = tt.Transformer(
    config=model_config,
    rng=tt.Random(MODEL_SEED),
)

parameter_count = sum(parameter.size for parameter in model.parameters())

print(model)
print("Parameter tensors:", len(model.parameters()))
print(f"Scalar parameters: {parameter_count:,}")

## 6. Configure training and callbacks

In [ ]:
loss_function = tt.CrossEntropyLoss()

optimizer = tt.AdamOptimizer(
    learning_rate=INITIAL_LEARNING_RATE,
    beta1=0.9,
    beta2=0.999,
    epsilon=1.0e-8,
    weight_decay=WEIGHT_DECAY,
)

training_config = tt.TrainingConfig()
training_config.epochs = EPOCHS
training_config.batch_size = BATCH_SIZE
training_config.device = DEVICE
training_config.log_every_steps = LOG_EVERY_STEPS
training_config.run_name = "comp_bpe_512_python_shakedown"
training_config.checkpoint_directory = str(CHECKPOINT_DIR)
training_config.enable_profiling = False
training_config.print_profile_summary = False
training_config.synchronize_profiling_phases = False
training_config.validate()

for path in CHECKPOINT_DIR.glob(f"{training_config.run_name}_*"):
    path.unlink()

generation_config = tt.GenerationConfig()
generation_config.max_new_tokens = GENERATION_MAX_NEW_TOKENS
generation_config.temperature = GENERATION_TEMPERATURE
generation_config.top_k = min(GENERATION_TOP_K, dataset.vocab_size)
generation_config.greedy_sampling = False
generation_config.print_generated_text = True
generation_config.print_token_ids = False
generation_config.random_seed = GENERATION_SEED
generation_config.validate()

trainer = tt.Trainer(
    model=model,
    loss_function=loss_function,
    optimizer=optimizer,
    config=training_config,
)

periodic_checkpoint = tt.CheckpointCallback(
    checkpoint_directory=str(CHECKPOINT_DIR),
    every_n_epochs=CHECKPOINT_EVERY_EPOCHS,
)

best_checkpoint = tt.BestCheckpointCallback(
    checkpoint_directory=str(CHECKPOINT_DIR),
    min_delta=EARLY_STOPPING_MIN_DELTA,
    prefer_validation_loss=True,
)

early_stopping = tt.EarlyStoppingCallback(
    patience=EARLY_STOPPING_PATIENCE,
    min_delta=EARLY_STOPPING_MIN_DELTA,
    prefer_validation_loss=True,
)

lr_scheduler = tt.LearningRateSchedulerCallback(
    schedule=tt.LearningRateSchedule.COSINE_DECAY,
    step_size=1,
    gamma=1.0,
    minimum_learning_rate=MINIMUM_LEARNING_RATE,
)

generation_callback = tt.GenerationCallback(
    tokenizer=tokenizer,
    prompt=GENERATION_PROMPT,
    generation_config=generation_config,
    every_n_epochs=GENERATION_EVERY_EPOCHS,
    print_generated_text=True,
    random_seed=GENERATION_SEED,
)

trainer.add_callback(periodic_checkpoint)
trainer.add_callback(best_checkpoint)
trainer.add_callback(early_stopping)
trainer.add_callback(lr_scheduler)
trainer.add_callback(generation_callback)

## 7. Run the 25-epoch shakedown

In [ ]:
training_start = time.perf_counter()

trainer.train(
    train_batches=train_batches,
    validation_batches=validation_batches,
)

training_elapsed = time.perf_counter() - training_start
history = trainer.history
completed_epochs = history.validation_count

print("\n" + "=" * 60)
print("Python Shakedown Results")
print("=" * 60)
print("Device:", DEVICE)
print("Completed epochs:", completed_epochs)
print("Training steps:", history.train_count)
print(f"Elapsed seconds: {training_elapsed:.3f}")
print(f"Seconds per epoch: {training_elapsed / max(1, completed_epochs):.3f}")
print("Initial training loss:", history.train_losses[0])
print("Final training loss:", history.train_losses[-1])
print("Initial validation loss:", history.validation_losses[0])
print("Final validation loss:", history.validation_losses[-1])
print("Initial learning rate:", lr_scheduler.initial_learning_rate)
print("Final learning rate:", lr_scheduler.current_learning_rate)
print("Best checkpoint epoch:", best_checkpoint.best_epoch)
print("Best checkpoint path:", best_checkpoint.best_checkpoint_path)
print("=" * 60)

## 8. Plot losses

In [ ]:
train_losses = np.asarray(history.train_losses, dtype=np.float32)
validation_losses = np.asarray(history.validation_losses, dtype=np.float32)
steps_per_epoch = len(train_batches)

epoch_train_losses = np.array(
    [
        train_losses[index:index + steps_per_epoch].mean()
        for index in range(0, len(train_losses), steps_per_epoch)
    ],
    dtype=np.float32,
)

plt.figure(figsize=(10, 6))
plt.plot(
    np.arange(1, len(epoch_train_losses) + 1),
    epoch_train_losses,
    marker="o",
    label="Mean training loss",
)
plt.plot(
    np.arange(1, len(validation_losses) + 1),
    validation_losses,
    marker="o",
    label="Validation loss",
)
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Transformer_Toy Edgar Allan Poe Shakedown")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 9. Review generation snapshots

In [ ]:
if not generation_callback.snapshots:
    print("No generation snapshots were captured.")
else:
    for snapshot in generation_callback.snapshots:
        print("\n" + "=" * 60)
        print(f"Generation snapshot — epoch {snapshot.epoch}")
        print("=" * 60)
        print(snapshot.text)

## 10. Load the best checkpoint and generate

In [ ]:
best_path = Path(best_checkpoint.best_checkpoint_path)

if not best_path.exists():
    raise FileNotFoundError(f"Best checkpoint was not created: {best_path}")

loaded_metadata = tt.CheckpointMetadata()
loaded_history = tt.TrainingHistory()

tt.Checkpoint.load(
    path=str(best_path),
    parameters=model.parameters(),
    metadata=loaded_metadata,
    history=loaded_history,
)

print("Loaded:", loaded_metadata)

final_generation_config = tt.GenerationConfig()
final_generation_config.max_new_tokens = 500
final_generation_config.temperature = 0.8
final_generation_config.top_k = 10
final_generation_config.greedy_sampling = False
final_generation_config.print_generated_text = True
final_generation_config.print_token_ids = False
final_generation_config.random_seed = 9876
final_generation_config.validate()

final_text = model.generate(
    prompt=GENERATION_PROMPT,
    tokenizer=tokenizer,
    config=final_generation_config,
    rng=tt.Random(9876),
)

print("\n" + "=" * 60)
print("Best-checkpoint generation")
print("=" * 60)
print(final_text)
print("=" * 60)

## 11. Save history

In [ ]:
history_path = ARTIFACT_ROOT / "training_history.csv"
history.save_csv(str(history_path))

print("History:", history_path)
print("Tokenizer:", TOKENIZER_PATH)
print("Checkpoints:", CHECKPOINT_DIR)

## Current limitations

- Dropout fields are set to zero because dropout and train/eval mode switching are planned for the next architectural milestone.
- Checkpoints restore model parameters, gradients, metadata, and history, but not Adam moment state.
- Batches are materialized before training to match the current native `Trainer` interface and make the run reproducible.